# ML Utilities

This notebook contains reusable helper functions for all Machine Learning models
used in the Log Guardian project.

Responsibilities:
- Import ML libraries
- Configure MLflow
- Define common constants
- Evaluate classification models
- Compare model performance
- Display feature importance
- Save and load trained models

This notebook does NOT train any machine learning models.

It is shared across:
- Binary Anomaly Detection
- Service Health Prediction
- Early Warning Prediction

In [0]:
## Necessary Imports
from pyspark.sql import functions as F
# ML Flow
import mlflow
import mlflow.spark

# Spark ML
from pyspark.ml import Pipeline

from pyspark.ml.classification import(
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

from pyspark.ml.feature import(
    StringIndexer,
    VectorAssembler,
    OneHotEncoder
)

from pyspark.ml.evaluation import(
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

print("ML Libraries Imported")

In [0]:
## MLflow Configuration

experiment_name = "/Shared/LogGuardian_ML"

mlflow.set_experiment(experiment_name)

print("Experiment:", experiment_name)

In [0]:
## Project Constants
## Random Seed
RANDOM_SEED = 42

## Train test Split
TRAIN_RATIO = 0.80
TEST_RATIO = 0.20

# Column Names
LABEL_COLUMN = "is_anomaly"
FEATURE_COLUMN = "features"
PREDICTION_COLUMN = "prediction"
PROBABILITY_COLUMN = "probability"
print("Project Constants Defined")

In [0]:
## Train Test Split Utility
def split_dataset(df):
    train_df, test_df = df.randomSplit(
        [TRAIN_RATIO, TEST_RATIO],
        seed=RANDOM_SEED
    )

    print(f"Training Rows : {train_df.count():,}")
    print(f"Testing Rows :{test_df.count():}")


    return train_df, test_df

In [0]:
## Evaluation Function
def evaluate_model(prediction):
    
    binary_eval = BinaryClassificationEvaluator(
        labelCol=LABEL_COLUMN,
        rawPredictionCol=PROBABILITY_COLUMN,
        metricName="areaUnderROC"
    )
    
    multi_eval = MulticlassClassificationEvaluator(
        labelCol=LABEL_COLUMN,
        predictionCol=PREDICTION_COLUMN
    )
    
    metrics = {
        "Accuracy":
        multi_eval.setMetricName("accuracy").evaluate(prediction),
        "Precision":
        multi_eval.setMetricName("weightedPrecision").evaluate(prediction),
        "Recall":
        multi_eval.setMetricName("weightedRecall").evaluate(prediction),
        "F1 Score":
        multi_eval.setMetricName("f1").evaluate(prediction),
        "ROC AUC":
        binary_eval.evaluate(prediction),
    }

    return metrics

In [0]:
## Display Metrics

def display_metrics(metrics):
    print("="*50)
    print("MODEL PERFORMANCE")
    print("="*50)

    for metric, value in metrics.items():
        print(f"{metric:<15}:{value:4f}")

In [0]:
## Model Comparison
def compare_models(results):
    """
    results should be
    [
        ("Random Forest", metrics),
        ("Decision Tree", metrics),
        ...    
    ]
    """
    comparison = []
    for model_name, metrics in results:
        comparison.append(
            (
                model_name,
                metrics["Accuracy"],
                metrics["Precision"],
                metrics["Recall"],
                metrics["F1 Score"],
                metrics["ROC AUC"]
            )
        )
    
    comparison_df = spark.createDataFrame(
        comparison,
        [
            "Model",
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score",
            "ROC AUC"
        ]
    )

    return comparison_df.orderBy(F.desc("F1 Score"))

In [0]:
## Feature Importance
def show_fearure_importance(model, feature_names):
    if hasattr(model, "featureImportance"):
        importance = list(model.featureImportances)
        importance_df = spark.createDataFrame(
            zip(feature_names, importance),
            ["Feature", "Importance"]
        )

        return importance_df.orderBy(
            F.desc("Importance")
        )

    else:
        print("Feature importance not available")

In [0]:
## Save Model
def save_model(model, model_path):
    model.write().overwrite().save(model_path)
    print(f"Model saved to {model_path}")

In [0]:
## Load Model
from pyspark.ml import PipelineModel

def load_model(model_path):
    model = PipelineModel.load(model_path)
    print("Model Loaded Successfully")

    return model

In [0]:
## Confusion Matrix
def confusion_matrix(prediction):
    cm = (
        prediction
        .groupBy(LABEL_COLUMN, PREDICTION_COLUMN)
        .count()
        .orderBy(LABEL_COLUMN, PREDICTION_COLUMN)
    )

    print("="*60)
    print("CONFUSION MATRIX")
    print("="*60)

    display(cm)

    return cm

In [0]:
## MLflow paramter logger
def log_parameters(params):
    """
    Logs model parameters to MLflow

    Paramters
    ----------
    params:dict
    """
    for key, value in params.items():
        mlflow.log_param(key, value)

In [0]:
## MLflow Metric Logger
def log_metrics(metrics):
    """
    Logs evaluation metrics to MLflow
    """
    for key, value in metrics.items():
        mlflow.log_metric(key, value)

In [0]:
## Log Model
def log_model(model, model_name):
    mlflow.spark.log_model(
        spark_model = model,
        artifact_path = model_name
    )

    print(f"{model_name} logged successfully.")

In [0]:
## Classification Report
def classification_report(predictions):
    metrics = evaluate_model(predictions)
    display_metrics(metrics)
    confusion_matrix(predictions)
    return metrics

In [0]:
## Feature Importance Plot
def top_features(model, feature_names, top_n=20):
    importance_df = show_feature_importance(
        model,
        feature_names
    )

    if importance_df:
        display(
            importance_df.limit(top_n)
        )

In [0]:
## Model Summary
def model_summary(
    model_name,
    metrics
):
    print("="*60)
    print(model_name.upper())
    print("="*60)

    for key, value in metrics.items():
        print(f"{key:<15}: {value:4f}")

In [0]:
## MLflow Run Helper
from contextlib import contextmanager

@contextmanager
def mlflow_run(run_name):
    with mlflow.start_run(run_name=run_name):
        print(f"Run Started : {run_name}")
        yield
        print("Run Completed")

In [0]:
UTILITY_VERSION = "1.0"
print(f"Ml Utilities version: {UTILITY_VERSION}")

In [0]:
print("="*70)
print("ML UTILITIES INITIALIZED")
print("="*70)

print("✔ MLflow Configured")
print("✔ Project Constants Defined")
print("✔ Train/Test Split Utility")
print("✔ Evaluation Utility")
print("✔ Model Comparison Utility")
print("✔ Feature Importance Utility")
print("✔ Confusion Matrix Utility")
print("✔ Classification Report Utility")
print("✔ MLflow Logging Utilities")
print("✔ Model Save/Load Utility")
print("✔ Utility Version:", UTILITY_VERSION)

print("="*70)
print("Status : SUCCESS")
print("="*70)